
# Galaxy broadband colors depend on the SSP library

The choice of SSP library propagates into the colors a photometric
fitter recovers — a single fixed galaxy SFH and dust law, rebuilt
with FSPS-MIST, FSPS-Padova/MILES, BPASS, BC03, and CB19 in turn,
produces a noticeable spread in NUV − r, u − g, g − r, and r − K.

The spread is largest in NUV − r (sensitive to recent O/B stars,
where BPASS binaries differ most) and in r − K (sensitive to the
TP-AGB treatment, where BC03 and FSPS-MIST differ most).

This is the systematic an SED fitter inherits from its assumed SSP
grid even before any prior or noise is involved.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

LIBRARIES = [
    ("fsps_prsc_miles_chabrier", "FSPS-Padova/MILES"),
    ("fsps_mist_c3k_a_chabrier", "FSPS-MIST/C3K"),
    ("bpss_stars_c3k_a_chabrier", "BPASS stars-only"),
    ("bc03_pdva_stelib_chabrier", "BC03/STELIB"),
    ("cb19_templates", "CB19"),
]

BANDS = ["galex_nuv", "sdss_u", "sdss_g", "sdss_r", "2mass_ks"]
COLORS_TO_PLOT = [
    ("NUV - r", 0, 3),
    ("u - g", 1, 2),
    ("g - r", 2, 3),
    ("r - K", 3, 4),
]

obs = tengri.Observation(photometry=tengri.Photometry.from_names(BANDS))

data = {}
first_failure: Exception | None = None
for ssp_name, label in LIBRARIES:
    try:
        ssp = tengri.load_ssp(ssp_name)
    except Exception as e:
        # Was `except (FileNotFoundError, Exception)`: the tuple reads as a
        # missing-grid check but the second member makes it a catch-all.
        if first_failure is None:
            first_failure = e
        continue
    sfh = {
        "type": "tsnorm",
        "all_params": tengri.FIXED,
        "peak_lbt_gyr": 3.0,
        "width_gyr": 2.0,
        "log_total_mass": 10.0,
        "skew": 0.3,
        "trunc": 10.0,
    }
    dust = {
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_diff": 0.2,
        "tau_bc": 0.3,
        "slope": -0.7,
    }
    model = tengri.SEDModel.build(
        ssp,
        observation=obs,
        sfh=sfh,
        dust=dust,
        redshift=tengri.Fixed(0.05),
    )
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    flux = np.asarray(model.predict_photometry(p))
    colors = {name: -2.5 * np.log10(flux[i] / flux[j]) for name, i, j in COLORS_TO_PLOT}
    data[label] = colors

if not data:
    raise RuntimeError(
        f"none of the {len(LIBRARIES)} SSP libraries loaded, so there are no "
        f"colors to compare. First failure: "
        f"{type(first_failure).__name__}: {first_failure}"
    ) from first_failure

fig, ax = plt.subplots(figsize=(7.2, 4.6))
x = np.arange(len(COLORS_TO_PLOT))
width = 0.16
colors_cmap = plt.cm.viridis(np.linspace(0.05, 0.92, len(data)))

for i, (label, color) in enumerate(zip(data, colors_cmap)):
    vals = [data[label][name] for name, _, _ in COLORS_TO_PLOT]
    ax.bar(
        x + (i - len(data) / 2) * width,
        vals,
        width,
        color=color,
        label=label,
        edgecolor="0.15",
        lw=0.4,
    )

ax.set_xticks(x)
ax.set_xticklabels([name for name, _, _ in COLORS_TO_PLOT])
ax.set_ylabel(r"AB color  [mag]")
ax.axhline(0.0, color="0.55", lw=0.6)
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2)
ax.text(
    0.97,
    0.05,
    "same SFH, dust, redshift —\nonly the SSP library changes",
    transform=ax.transAxes,
    ha="right",
    fontsize=8,
    color="0.4",
)

fig.tight_layout()
plt.savefig("plot_ssp_color_compare.png", dpi=150, bbox_inches="tight")